In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt

# Add the project root to the sys.path to import from src
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from src.data_loader import load_record
from src.preprocessing import filter_ecg
from src.feature_extraction import extract_rr_sequence
from src.hmm_model import train_hmm, compute_likelihood, compute_threshold
from src.visualization import plot_results 

: 

In [ ]:
record_id = "100"  # Normal record
signal, r_peaks, fs = load_record(record_id)

# Filter the signal
filtered_signal = filter_ecg(signal, fs)

# Plot a 5-second window comparison
plt.figure(figsize=(15, 5))
time = np.arange(len(signal)) / fs
limit = int(5 * fs)

plt.plot(time[:limit], signal[:limit], label="Raw ECG", alpha=0.5)
plt.plot(time[:limit], filtered_signal[:limit], label="Filtered ECG (0.5-45Hz)", color='teal')
plt.title(f"Record {record_id} - Signal Preprocessing")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.legend()
plt.show()

In [ ]:
rr = extract_rr_sequence(r_peaks, fs)

plt.figure(figsize=(12, 4))
plt.plot(rr, marker='o', linestyle='--', color='dodgerblue')
plt.axhline(np.mean(rr), color='red', linestyle='-', label=f"Mean: {np.mean(rr):.2f}s")
plt.title("RR Interval Sequence (Heartbeat Timings)")
plt.xlabel("Beat Index")
plt.ylabel("Time (seconds)")
plt.legend()
plt.show()

In [ ]:
training_ids = ["100", "101"]
training_sequences = []

for rid in training_ids:
    _, peaks, freq = load_record(rid)
    seq = extract_rr_sequence(peaks, freq)
    training_sequences.append(seq)

# Train HMM with 4 hidden states
model, train_lls = train_hmm(training_sequences, n_components=4)

# Compute threshold (Mean - 2*Std)
threshold, mean_ll, std_ll = compute_threshold(train_lls, n_std=2.0)

print(f"\nFinal Anomaly Threshold: {threshold:.4f}")

In [ ]:
test_record = "108" # Known for PVCs
sig_test, peaks_test, fs_test = load_record(test_record)
filtered_test = filter_ecg(sig_test, fs_test)
rr_test = extract_rr_sequence(peaks_test, fs_test)

# Score the sequence
ll_score = compute_likelihood(model, rr_test)
label = "Normal" if ll_score >= threshold else "Possible Arrhythmia"

# Use the project's visualization module
plot_results(
    signal=filtered_test,
    r_peaks=peaks_test,
    rr_intervals=rr_test,
    fs=fs_test,
    record_id=test_record,
    log_likelihood=ll_score,
    threshold=threshold,
    label=label
)